In [11]:
# Notebook 04 - APRENDIZADO DE MÁQUINA

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Carregar o dataset processado df_mega.csv
caminho_mega = "../data/processed/df_mega.csv"
if not os.path.exists(caminho_mega):
  raise FileNotFoundError(
      "Execute o Notebook 03 primeiro para gerar o df_mega.csv!"
  )

df_mega = pd.read_csv(caminho_mega)

# 2. Separar Features (X), Rótulo (y) e Grupos (sujeito_id)
features = ["BCEA", "Var_X", "Var_Y", "Mean_X", "Mean_Y"]
X = df_mega[features]
y = df_mega["target"]
groups = df_mega["sujeito_id"]

print(
    f"Matriz de features carregada. Formato: {X.shape}. Total de sujeitos"
    f" únicos: {groups.nunique()}"
)

# 3. Configurar Validação Cruzada por Grupo (GroupKFold)
# Garante que dados de um mesmo sujeito nunca fiquem divididos entre treino e teste
gkf = GroupKFold(n_splits=5)

acuracias = []

print("\nIniciando validação cruzada rigorosa por sujeito...")
for fold, (treino_idx, teste_idx) in enumerate(gkf.split(X, y, groups)):
  X_train, X_test = X.iloc[treino_idx], X.iloc[teste_idx]
  y_train, y_test = y.iloc[treino_idx], y.iloc[teste_idx]

  # Escalonamento dos dados
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  # Treinamento do modelo
  modelo = RandomForestClassifier(n_estimators=100, random_state=42)
  modelo.fit(X_train_scaled, y_train)

  # Predição
  y_pred = modelo.predict(X_test_scaled)

  # Avaliação do fold
  acc = accuracy_score(y_test, y_pred)
  acuracias.append(acc)
  print(f"Fold {fold+1} - Acurácia: {acc:.4f}")

print("\n--- RESULTADO GERAL DA VALIDAÇÃO ---")
print(f"Acurácia Média por Sujeito: {np.mean(acuracias)*100:.2f}%")

Matriz de features carregada. Formato: (457, 5). Total de sujeitos únicos: 41

Iniciando validação cruzada rigorosa por sujeito...
Fold 1 - Acurácia: 0.5000
Fold 2 - Acurácia: 0.5978
Fold 3 - Acurácia: 0.4022
Fold 4 - Acurácia: 0.4348
Fold 5 - Acurácia: 0.6703

--- RESULTADO GERAL DA VALIDAÇÃO ---
Acurácia Média por Sujeito: 52.10%
